# AgriMind — Full project workflow (Google Colab)

**For submission:** use **Runtime → Run all**, wait until every cell finishes, then **File → Download → Download .ipynb** so your teacher sees **code and outputs** together.

This notebook mirrors the local project:
- Real APY crop data + state climate merge  
- Preprocessing and feature engineering  
- Yield models (Linear, HGB, Tree, RF, XGB) with **metrics table**  
- Full `run_all` pipeline (artifacts + crop classifier + SHAP plot)  
- Sample prediction  

**Repo folder on Colab:** usually `/content/ml-project-main/backend` (GitHub ZIP) or `/content/MLproject/backend` (git clone).

## 1) Install packages (same stack as backend ML pipeline)

In [ ]:
%%capture
!pip install -q numpy pandas scikit-learn xgboost joblib shap matplotlib seaborn

## 2) Point Colab at your **backend** folder

After a **new runtime**, `/content` is empty except `sample_data` — you must either:
- **Set `GITHUB_USER` + `REPO_NAME`** below and let the cell **download + unzip** your repo, or  
- Upload your project ZIP to `/content` and unzip (then re-run this cell).

The cell **searches all of `/content`** for a folder that contains both `app.py` and `ml_models/`.

In [ ]:
%matplotlib inline
import os, sys
import urllib.request
import zipfile
from pathlib import Path

# If backend not found, download public GitHub repo as ZIP (edit these):
GITHUB_USER = ""  # e.g. "your-github-username"
REPO_NAME = ""    # e.g. "ml-project"
BRANCH = "main"


def find_backend():
    flat = [
        Path("/content/ml-project-main/backend"),
        Path("/content/MLproject/backend"),
        Path("/content/MLproject-main/backend"),
        Path("/content/backend"),
        Path.cwd(),
    ]
    for c in flat:
        try:
            if c.is_dir() and (c / "ml_models").is_dir() and (c / "app.py").is_file():
                return c.resolve()
        except OSError:
            pass
    root = Path("/content")
    if root.is_dir():
        for child in sorted(root.iterdir()):
            if not child.is_dir() or child.name.startswith("."):
                continue
            for sub in (child / "backend", child):
                if (
                    sub.is_dir()
                    and (sub / "ml_models").is_dir()
                    and (sub / "app.py").is_file()
                ):
                    return sub.resolve()
        for app in root.rglob("app.py"):
            d = app.parent
            if (d / "ml_models").is_dir():
                return d.resolve()
    return None


BACKEND = find_backend()

if BACKEND is None and GITHUB_USER and REPO_NAME:
    root = Path("/content")
    zip_path = root / "repo_source.zip"
    url = f"https://github.com/{GITHUB_USER}/{REPO_NAME}/archive/refs/heads/{BRANCH}.zip"
    print("Downloading:", url)
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(root)
    BACKEND = find_backend()

if BACKEND is None:
    raise FileNotFoundError(
        "No backend found (need app.py + ml_models/).\n"
        "Folders in /content: " + str([p.name for p in Path('/content').iterdir()]) + "\n"
        "Fix: set GITHUB_USER and REPO_NAME in this cell (public repo), or upload/unzip your project under /content, then re-run."
    )

os.chdir(BACKEND)
sys.path.insert(0, str(BACKEND.resolve()))
os.environ["MPLBACKEND"] = "inline"
print("BACKEND:", BACKEND.resolve())

## 3) Download crop production CSV (if missing)

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
csv_path = DATA_DIR / "crop_production_india.csv"
URL = "https://raw.githubusercontent.com/vish-manit/Crop-Production-Statistics-in-India/master/APY.csv"
MAX_ROWS = 50000

if not csv_path.is_file():
    df_dl = pd.read_csv(URL, low_memory=False)
    if MAX_ROWS and len(df_dl) > MAX_ROWS:
        df_dl = df_dl.sample(n=MAX_ROWS, random_state=42).reset_index(drop=True)
    df_dl.to_csv(csv_path, index=False)
    print("Saved rows:", len(df_dl))
else:
    df_dl = pd.read_csv(csv_path, nrows=5)
    print("CSV already present; showing 5-row peek:")
    display(df_dl)

## 4) Load pipeline data + **preview** (outputs for report)

In [ ]:
from IPython.display import display
from ml_models.pipeline.data_loader import load_dataset
from ml_models.pipeline.preprocess import clean_raw_frame
from ml_models.pipeline.features import add_yield_and_rainfall_category

raw, src = load_dataset(verbose=True)
print("Source:", src, "| Raw rows:", len(raw))
clean = clean_raw_frame(raw)
print("After clean:", len(clean))
feat = add_yield_and_rainfall_category(clean)
print("After yield filter:", len(feat))
print("Yield mean / std:", round(feat["yield"].mean(), 3), round(feat["yield"].std(), 3))
display(feat.head(10))
display(feat[["yield", "rainfall", "temperature", "area"]].describe())

## 5) **EDA charts** (inline — visible in saved .ipynb)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

s = feat.sample(min(4000, len(feat)), random_state=42)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(s["rainfall"], s["yield"], alpha=0.35, s=14, c="#0d9488")
axes[0].set_xlabel("Rainfall (mm)")
axes[0].set_ylabel("Yield (t/ha)")
axes[0].set_title("Yield vs rainfall (sample)")
top = feat["crop"].value_counts().head(15)
top.plot(kind="bar", ax=axes[1], color="#6366f1")
axes[1].set_title("Top 15 crops by row count")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 6) **Model comparison table** (R², RMSE, MAE)
Uses a random subsample so this cell finishes in reasonable time; same estimators as `train_regressors.py`.

In [ ]:
from ml_models.pipeline.train_regressors import train_and_compare

DEMO_N = min(20000, len(feat))
demo_df = feat.sample(n=DEMO_N, random_state=42).reset_index(drop=True)
best_pipe_demo, table, meta = train_and_compare(demo_df)
display(table.round(4))
print("Best model (by test R²):", meta["best_model_name"])

## 7) **Full pipeline** — same as local `python -m ml_models.pipeline.run_all`
Writes `yield_xgb.joblib`, crop model, EDA PNGs, SHAP summary. **Scroll this output** for crop classifier accuracy and full logs.

In [ ]:
import os
os.environ["AGRIMIND_MAX_TRAIN_ROWS"] = "50000"
os.environ.pop("AGRIMIND_USE_SYNTHETIC_DATA", None)

from ml_models.pipeline.run_all import main
main()

## 8) **Saved figures** from disk (EDA + SHAP)

In [ ]:
from IPython.display import Image, display
from pathlib import Path

eda = Path("ml_models/artifacts/eda")
for name in ["yield_vs_rainfall.png", "crop_distribution.png", "correlation_heatmap.png", "shap_summary_yield.png"]:
    p = eda / name
    if p.is_file():
        print(name)
        display(Image(filename=str(p)))
    else:
        print("(missing)", p)

## 9) **Inference** — single prediction with saved pipeline

In [ ]:
import joblib
import pandas as pd

pipe = joblib.load("ml_models/artifacts/yield_xgb.joblib")

def rainfall_cat(r):
    r = float(r)
    if r < 500: return "low"
    if r <= 1000: return "medium"
    return "high"

examples = [
    {"state": "maharashtra", "crop": "rice", "season": "kharif", "rainfall": 1100, "temperature": 27, "area": 2},
    {"state": "punjab", "crop": "wheat", "season": "rabi", "rainfall": 650, "temperature": 22, "area": 5},
]
for ex in examples:
    ex = dict(ex)
    ex["rainfall_category"] = rainfall_cat(ex["rainfall"])
    row = pd.DataFrame([ex])
    y = float(pipe.predict(row)[0])
    print(ex["state"], ex["crop"], "→ predicted yield (t/ha):", round(y, 3))

---
### Checklist before submitting
- [ ] **Runtime → Run all** completed without errors  
- [ ] **File → Download .ipynb** (with outputs)  
- [ ] Section **6** shows the metrics **table**  
- [ ] Section **7** shows training log + crop accuracy  
- [ ] Section **8** shows at least some images  

*Note: Section 6 and Section 7 both train models (demo table vs full artifact export). For a faster run, increase `DEMO_N` only in reports or reduce `AGRIMIND_MAX_TRAIN_ROWS`.*